In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [6]:
matmul=r"""

#include <cstdio>
#include <cuda_runtime.h>

#define TILE 16

__global__ void matmul_tiled(const float* A, const float* B, float* C, int M, int N, int K){

    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int tx=threadIdx.x;
    int ty=threadIdx.y;
    int col = blockIdx.x*blockDim.x+tx;
    int row = blockIdx.y*blockDim.y+ty;

    float sum=0.0f;

    int numTiles = (K+TILE-1)/TILE;
    
    for (int t=0; t<numTiles; t++){
        int aCol=t*TILE+tx;
        int bRow=t*TILE+ty;

        As[ty][tx] = (row < M && aCol < K) ? A[row * K + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < K && col < N) ? B[bRow * N + col] : 0.0f;
        __syncthreads();

        for (int k=0; k<TILE; k++){
            sum+=As[ty][k]*Bs[k][tx];
        }
        __syncthreads();
    }

    if (row<M && col<N){
        C[row*N+col]=sum;
    }
}

int main(){
    const int M=100, K=200, N=300;
    
    float *A, *B, *C;
    cudaMallocManaged(&A, M*K*sizeof(float));
    cudaMallocManaged(&B, K*N*sizeof(float));
    cudaMallocManaged(&C, M*N*sizeof(float));

    for(int i=0; i<M*K; i++) A[i]=1.0f;
    for(int i=0; i<K*N; i++) B[i]=2.0f;

    dim3 block(TILE, TILE);
    dim3 grid((N+TILE-1)/TILE, (M+TILE-1)/TILE);
    
    matmul_tiled<<<grid, block>>>(A, B, C, M, N, K);
    cudaDeviceSynchronize();

    printf("C[0][0] = %f\n", C[0]);
    printf("C[%d][%d] = %f\n", M-1, N-1, C[(M - 1) * N + (N - 1)]);

    cudaFree(A);
    cudaFree(B);
    cudaFree(C);

    return 0;
}

"""

with open('matmul.cu', 'w') as f:
    f.write(matmul)

In [7]:
!nvcc -arch=sm_75 matmul.cu -o matmul

In [8]:
!./matmul

C[0][0] = 400.000000
C[99][299] = 400.000000
